# 05 Models B, C and C2 under the final tuning rule v3

Post-hoc changes of 2026-11-18 (CLAUDE.md Decision Log): rule v3 = v2 plus a levelling-down guard (every group's inner-fold stratified minor TPR must be at least A's - 0.02), frozen as the final tuning rule; C2's DRO weights capped to 0.1-10x each group's natural share, eta grid {0.001, 0.01, 0.1}. The v2 B, C, C2 files are archived on Drive first (sha256 checked) and stay in the repo under results/models/v2/. GPU: T4, about 1 hour. Charts only; no images are opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions and archive of the v2 runs

In [ ]:
A = json.loads((DRIVE_ROOT / 'results/models/A.json').read_text())
assert A['git']['dirty'] is False
assert sha256(DRIVE_ROOT / A['oof_file']['drive_path']) == A['oof_file']['sha256'], "A's OOF file changed"
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
import shutil
V2_COMMIT = 'cfab668'
R, P = DRIVE_ROOT / 'results', DRIVE_ROOT / 'per_image'
(R / 'models/v2').mkdir(parents=True, exist_ok=True); (P / 'v2_archive').mkdir(parents=True, exist_ok=True)
for m in ['B', 'C', 'C2']:
    dst = R / f'models/v2/{m}.json'
    if dst.exists():
        print(m, 'v2 already archived'); continue
    d = json.loads((R / f'models/{m}.json').read_text())
    assert d['git']['commit'].startswith(V2_COMMIT) and d['select'] == 'worst_group_v2', f'{m}: not the v2 run'
    oof = DRIVE_ROOT / d['oof_file']['drive_path']
    assert sha256(oof) == d['oof_file']['sha256'], f'{m} v2 OOF changed'
    shutil.copy(oof, P / 'v2_archive' / oof.name)
    assert sha256(P / 'v2_archive' / oof.name) == d['oof_file']['sha256']
    shutil.copy(R / f'models/{m}.json', dst)
    print(m, 'v2 archived (JSON + OOF, sha256 checked)')
if not (R / 'models/v2/comparisons.json').exists():
    shutil.copy(R / 'comparisons.json', R / 'models/v2/comparisons.json')

## 2. Model B, rule v3 (~5 min)

In [ ]:
!python scripts/train_model.py --model B
d = save_manifest('models/B.json')
for f in d['folds']:
    print(f['test_fold'], f['chosen'], f['epochs'], 'WG', round(f['inner_val_worst_group_tpr'], 3), 'allowed', f['inner_val_feasible'], 'test MAE', round(f['test_mae'], 2), 'ref A MAE diff', round(f['reference_A']['mae_diff_vs_A_record'], 4))

## 3. Model C, rule v3 (~25 min)

In [ ]:
!python scripts/train_model.py --model C
d = save_manifest('models/C.json')
for f in d['folds']:
    print(f['test_fold'], f['chosen'], f['epochs'], 'WG', round(f['inner_val_worst_group_tpr'], 3), 'allowed', f['inner_val_feasible'], 'test MAE', round(f['test_mae'], 2), 'ref A MAE diff', round(f['reference_A']['mae_diff_vs_A_record'], 4))
    print('   top weights', f['refit_info'].get('dro_final_weights_top5'))

## 4. Model C2 with capped weights, rule v3 (~20 min)

In [ ]:
!python scripts/train_model.py --model C2
d = save_manifest('models/C2.json')
for f in d['folds']:
    print(f['test_fold'], f['chosen'], f['epochs'], 'WG', round(f['inner_val_worst_group_tpr'], 3), 'allowed', f['inner_val_feasible'], 'test MAE', round(f['test_mae'], 2), 'ref A MAE diff', round(f['reference_A']['mae_diff_vs_A_record'], 4))
    print('   weight/share (cap 10)', f['refit_info'].get('dro_weight_over_share_top5'))

## 5. Holm-adjusted verdicts vs A (interim, family of 9)

In [ ]:
!python scripts/compare_models.py
save_manifest('comparisons.json')

## 6. Aggregate outputs for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step9_v3_outputs.zip'
files = [DRIVE_ROOT / f'results/models/{m}.json' for m in ['B', 'C', 'C2']] + [DRIVE_ROOT / 'results/comparisons.json']
with zipfile.ZipFile(out, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(DRIVE_ROOT))
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))